# Building a Specialized Model: Iterative Workflow

This guide covers how to build a highly accurate, domain-specific language model when no ready-made training data exists for your target — and how to improve a model you have already trained but are not satisfied with.

---

## Who this is for

This guide is for two groups:

1. **Starting from nothing** — you need a model for a domain, language variety, or historical period where no UD-annotated treebank exists and no suitable base model is available off the shelf.
2. **Improving an existing model** — you have already trained a model using the main tutorial but accuracy is not where you need it, and you want to improve it.

In both cases the solution is the same: **you get better models by getting more and better annotated training data.** The iterative workflow is the fastest way to build that data — quicker than annotating everything by hand before any training.

---

## The strategy

Instead of annotating a large corpus before training anything, you start small, train a first model, use it to pre-annotate new text, correct the pre-annotations, and fine-tune. Each round, the model gets better — and better model predictions mean faster corrections.

```text
[Optional: related UD data] ──► Train base model  ──► evaluate on held-out test (FINAL ONLY)
                                        │
                         ┌──────────────▼──────────────────┐
                         │  Auto-annotate new texts         │
                         │  Correct predictions by hand     │  ◄── repeat
                         │  Fine-tune model (main tutorial) │
                         └──────────────────────────────────┘
                                    ▲
                              track progress
                              on dev set only
```

The end product is both a growing corpus of high-quality annotated data **and** an increasingly accurate fine-tuned model. Each iteration makes the model better.

## Prerequisites

The `lexos` package must be installed in your Python environment before running this notebook. If you haven't done this yet, run the following in a terminal from the repository root:

```bash
pip install -e .
```

This installs the package in editable mode so any changes you make to the source are reflected immediately. If you are a regular user (not modifying the source), install from PyPI once the package is published:

```bash
pip install lexos
```

After installation, run the import cell below.

In [ ]:
from pathlib import Path

from lexos.language_model import export_to_conllu, combine_conllu

## Configuration

The cell below sets the working directory used throughout this notebook.

**`_tutorial_dir`** — the folder where your model and annotation data live. By default it is derived automatically from the installed package location, which works if you are running this notebook from inside the Lexos repo after `pip install -e .`.

If you are working outside the repo — for example, running this notebook in your own project folder — replace the auto-derivation with a direct path:

```python
_tutorial_dir = Path("C:/Users/you/my_project")  # Windows
_tutorial_dir = Path("/home/you/my_project")       # Mac / Linux
```

The cells in this notebook use `_tutorial_dir` as a base for reading `.txt` source files and writing CONLL-U output files.

In [ ]:
import lexos.language_model as _lm

# Derive the tutorial folder from the installed package path.
# Works when running from the Lexos repo with pip install -e .
#   __init__.py → language_model/ → lexos/ → src/ → repo root
_repo_root = Path(_lm.__file__).parents[3]
_tutorial_dir = _repo_root / "src" / "docs" / "tutorials" / "language_model"

# Not in the Lexos repo? Replace the two lines above with a direct path:
# _tutorial_dir = Path("C:/Users/you/my_project")

print(f"Working directory: {_tutorial_dir}")
if not _tutorial_dir.exists():
    print("WARNING: Directory not found — update _tutorial_dir above.")

---

## Phase 1 — Get your first round of training data

You need some annotated sentences before you can train anything.

### Option A — Start with a related UD treebank (optional accelerator)

If a UD treebank exists for a related dialect, variety, or domain, it can give your first model a head start. See [Getting Training Data](../../user_guide/language_model/training_data.md) for where to find and download treebanks.

**How related is related enough?** Language variants transfer best (Early Modern English → Modern English). Domain shift is next best (legal text → general English). Cross-family data (unrelated language) may hurt more than it helps.

If no related data exists, go directly to Option B. A related treebank is a convenience, not a requirement.

### Option B — Annotate your first batch by hand

Annotate 200–500 sentences in your target domain from scratch. See [Getting Training Data](../../user_guide/language_model/training_data.md) for annotation tools, UD guidelines, and format details.

For tool recommendations: [INCEpTION](https://inception-project.github.io/) and [Arborator Grew](https://arboratorgrew.elizia.net/) are free and support UD natively. [Prodigy](https://prodi.gy/) (commercial, from Explosion — creators of spaCy) is particularly fast once you have a model to pre-annotate with.

**Quality matters more than quantity.** Read the [Annotation quality](#annotation-quality) section before you start.

---

## Phase 2 — Train your first base model

**For this step, follow the [main tutorial](tutorial.ipynb).**

Pass the following to the tutorial:

- **`data`** — your Phase 1 training data (related treebank, your hand annotations, or both combined with `combine_conllu`)
- **`base_model`** — the best UD-trained model available for your language. If nothing suitable exists, omit `base_model` entirely (or omit only the components with no suitable source) — any component not specified is trained from scratch.

After training, you will have a `model-best` directory. This is your starting point for Phase 3.

---

## Phase 3 — Bootstrap: auto-annotate and correct

This is the core of the iterative workflow. Use your trained model to pre-annotate new text, then correct the output by hand. Correcting pre-annotations is significantly faster than annotating from scratch — and it gets faster each round as the model improves.

### Step 1 — Export model annotations to CONLL-U

The cell below runs your model on new text and writes the predictions to a CONLL-U file ready for correction.

**What to update before running:**

- `_model_path` — path to your current `model-best` directory (update each round to the latest model)
- `_output_path` — output filename (use a new name each round: `round1_auto.conllu`, `round2_auto.conllu`, …)
- Choose one of the three **load texts** patterns:
  - **Pattern 1** (default) — single `.txt` file; good for an initial test run
  - **Pattern 2** — all `.txt` files from a folder; use this for real annotation rounds
  - **Pattern 3** — literal sentences; fastest way to verify the output format before committing to a full batch

In [ ]:
# ── Round configuration — update these for each bootstrap round ──────────────
_model_path = str(_tutorial_dir / "winter_tale" / "training" / "en" / "model-best")
# ↑ Path to your current model-best — update each round

_output_path = _tutorial_dir / "round1_auto.conllu"
# ↑ Output file name — rename for each round (round1_auto, round2_auto, …)
# ─────────────────────────────────────────────────────────────────────────────

# ── Load texts — choose one of the patterns below ────────────────────────────

# Pattern 1: single file (good for an initial test run)
texts = [(_tutorial_dir / "Adv_tutorial_sample.txt").read_text(encoding="utf-8")]

# Pattern 2: all .txt files from a folder (common for a full annotation round)
# _texts_dir = _tutorial_dir / "phase3_texts"
# texts = [p.read_text(encoding="utf-8") for p in sorted(_texts_dir.glob("*.txt"))]

# Pattern 3: literal sentences (fastest way to verify output format)
# texts = ["Your first sentence.", "Your second sentence."]
# ─────────────────────────────────────────────────────────────────────────────

if not texts:
    raise ValueError("texts is empty — check that your file path or folder exists.")

export_to_conllu(
    model_path=_model_path,
    texts=texts,
    output_path=_output_path,
)

---

## ⚠️ STOP HERE — go annotate

Open the output file (`round1_auto.conllu` in your working directory, or whatever you set `_output_path` to) in your annotation tool and correct the predictions. Come back to this notebook when you have saved the corrected file.

> **Check the output before correcting.** Open the `.conllu` file in a text editor and confirm: sentences are separated by blank lines, each line has 10 tab-separated columns, and head indices are small integers that stay within each sentence's token range. If every token shows `head = 0` or the whole file is one block, check that `_model_path` points to the correct `model-best` directory.

### Step 2 — Correct the output

**In INCEpTION:**

1. Create a new project and select "Universal Dependencies" as the annotation layer
2. Import the CONLL-U file via *Documents → Import*
3. Review each sentence. The dependency tree is shown graphically — drag arcs to change the head, click labels to change the relation
4. Export via *Documents → Export → CoNLL-U*

**In Arborator Grew:**

1. Create a new project and import the CONLL-U file
2. The dependency tree is displayed as an interactive graph — click tokens and arcs to make corrections
3. Export to CONLL-U when done

**In Prodigy** (if you have a licence): Prodigy's `dep.correct` recipe wraps the entire loop — it loads your model, shows one sentence at a time with the predicted tree, and lets you accept or correct before moving to the next. This is the fastest path for large batches.

**What to look for when correcting:**

- **HEAD** (column 7) — is the dependency arc pointing to the right governor?
- **DEPREL** (column 8) — is the relation label correct? (`nsubj`, `obj`, `case`, `det`)
- **UPOS** (column 4) — is the universal POS tag right? (`NOUN`, `VERB`, `ADP`, etc.)
- **FEATS** (column 6) — are morphological features correct? (`Number=Sing`, `Tense=Past`, etc.)

Refer to the UD guidelines at [https://universaldependencies.org/guidelines.html](https://universaldependencies.org/guidelines.html) when you are unsure.

### Annotation quality

> **Annotation quality is the ceiling on your model quality.**
> Every error in your training data can become a systematic model error — and errors compound across rounds.

- **Better to annotate 100 sentences carefully than 300 carelessly.** Speed is a trap.
- **Be consistent.** The same construction should be annotated the same way every time.
- **When in doubt, look it up.** Consult the UD guidelines rather than guessing.

### How many sentences per round

| Round | Batch size | Notes |
| --- | --- | --- |
| 1 | 200–500 sentences | Predictions are rough; corrections are slow; start small |
| 2 | ~double Round 1 | Model improves; corrections get faster |
| 3+ | ~double each round | Scale with how fast corrections are going |

> **Critical discipline: never use the held-out test set during iteration.** Use the dev set to track progress round-to-round. Reserve the test set for your final evaluation.

---

## Phase 4 — Fine-tune

**For this step, follow the [main tutorial](tutorial.ipynb).**

Pass the following to the tutorial:

- **`data`** — the combined training file produced by Phase 5 below (after Round 1 this is `combined_training.conllu`; for Round 1 it is just your corrected file directly)
- **`base_model`** — all five components sourced from the previous round's `model-best`:

```python
base_model={
    "tok2vec":              "path/to/previous-round/model-best",
    "tagger":               "path/to/previous-round/model-best",
    "morphologizer":        "path/to/previous-round/model-best",
    "trainable_lemmatizer": "path/to/previous-round/model-best",
    "parser":               "path/to/previous-round/model-best",
}
```

After training, you have a new `model-best`. Update `_model_path` above to this new path for the next bootstrap round, then loop back to Phase 3.

---

## Phase 5 — Combine rounds

Before fine-tuning each round (after Round 1), combine all corrected annotation files from every previous round into a single training file. Training on the full accumulated corpus each round — not just the latest batch — produces more stable models.

Update `_round_files` below after each round by uncommenting the next line, then run this cell before going back to the main tutorial.

In [ ]:
# ── Update this list after each round of annotation ─────────────────────────
_round_files = [
    _tutorial_dir / "round1_corrected.conllu",   # ← rename to match your actual corrected file
    # _tutorial_dir / "round2_corrected.conllu",  # uncomment when round 2 is done
    # _tutorial_dir / "round3_corrected.conllu",  # and so on
]
_combined_output = _tutorial_dir / "combined_training.conllu"
# ─────────────────────────────────────────────────────────────────────────────

combined = combine_conllu(
    round_files=_round_files,
    output_path=_combined_output,
)
print(f"\nNext: run the main tutorial with data pointing to: {combined}")

---

## When to stop

Track accuracy on your **dev set** after each fine-tune round. spaCy reports LAS (Labelled Attachment Score) at the end of training — this is the primary metric for dependency parsing quality.

| LAS on dev | Interpretation |
| --- | --- |
| < 60% | Weak; more data or better annotation quality needed |
| 60–75% | Reasonable for specialised text; continue if time allows |
| 75–85% | Good; typical for well-resourced domains |
| 85%+ | Strong; marginal gains per annotation hour are small |

**Stop when** any of these apply:

- Dev LAS has not improved across two consecutive fine-tune rounds
- Corrections per hour are no longer increasing (model predictions are already accurate enough)
- You have reached your annotation time budget

After deciding to stop, run the main tutorial's evaluation step against your **held-out test set** to get your final honest accuracy number.